# Rizzler V24 — Live Production Qualification (Kaggle)
This notebook checks the exact repository revision and uses a paid Gemini API key supplied through Kaggle Secrets. It runs offline regressions separately from live work, records real artifacts and source SHA, and exports an honest evidence ZIP. **Passing does not itself prove every visual or factual result is production quality.**

In [ ]:
import os, sys, json, subprocess, pathlib, hashlib, time, secrets, shutil
ROOT=pathlib.Path("/kaggle/working")
REPO=ROOT/"Rizzler-V24"
BRANCH="feature/v24-context-convergence-and-safe-repair"
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,"https://github.com/SAUNAK359/Voice-AI.git",str(REPO)],check=True)
SHA=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
print("Pinned source SHA:",SHA)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[work,test]","pymupdf>=1.25,<2"],check=True)
sys.path.insert(0,str(REPO/"src"))
OUT=ROOT/"rizzler_v24_evidence"
OUT.mkdir(parents=True,exist_ok=True)
(OUT/"source_sha.txt").write_text(SHA+"\n")
print("Kaggle environment ready. Internet must be enabled; no key is printed.")

## Offline regression isolation
Run authorization, context recovery and existing specialist contract tests without live credentials. These are source-code tests, not an authenticated Gemini qualification.

In [ ]:
import copy
TEST_ENV={k:v for k,v in os.environ.items() if not k.startswith("RIZZLER_") and k not in ("GEMINI_API_KEY","GOOGLE_API_KEY")}
TEST_ENV.update(PYTHONPATH=str(REPO/"src"))
tests=["test_v24_context_convergence.py","test_v24_intent_and_recovery.py","test_v23_work_studio.py","test_v23_outcome_contract.py","test_v23_sql_exploration.py"]
runs={}
for filename in tests:
    p=subprocess.run([sys.executable,"-m","unittest","discover","-s",str(REPO/"tests"),"-p",filename,"-v"],cwd=REPO,env=TEST_ENV,text=True,capture_output=True)
    (OUT/(filename+".log")).write_text(p.stdout+"\n"+p.stderr)
    runs[filename]={"exit_code":p.returncode,"passed":p.returncode==0}
print(json.dumps(runs,indent=2))
(OUT/"offline_checks.json").write_text(json.dumps(runs,indent=2))

## Gemini authenticated preflight
Add Kaggle Secret named **GEMINI_API_KEY**. Set optional **RIZZLER_GEMINI_MODEL** as Kaggle environment variable to choose a verified model; otherwise use the repository's configured default. No static model ID is assumed for billing.

In [ ]:
from kaggle_secrets import UserSecretsClient
from google import genai
from rizzler.config import Settings
key=UserSecretsClient().get_secret("GEMINI_API_KEY")
if not key: raise RuntimeError("Missing Kaggle Secret: GEMINI_API_KEY")
os.environ["GEMINI_API_KEY"]=key
os.environ["RIZZLER_WORK_ROOT"]=str(OUT/"work")
os.environ["RIZZLER_ACCESS_KEY"]=secrets.token_urlsafe(32)
os.environ["RIZZLER_WORK_SIGNING_KEY"]=secrets.token_urlsafe(48)
os.environ["RIZZLER_V23_MODE"]="full"
os.environ["RIZZLER_CONTEXT_CONVERGENCE"]="1"
os.environ["RIZZLER_GEMINI_NATIVE_TOOLS"]="1"
os.environ["RIZZLER_GEMINI_ENABLE_THINKING"]="1"
os.environ["RIZZLER_ARTIFACT_QA"]="strict"
cfg=Settings.from_env()
client=genai.Client(api_key=key)
models=client.models.list()
names=[m.name for m in models]
model_short=cfg.gemini_model
if not any(n.endswith("/"+model_short) or n==model_short for n in names):
    raise RuntimeError("Configured Gemini model is not visible to this paid API key: "+model_short)
rsp=client.models.generate_content(model=model_short,contents="Return exactly: RIZZLER_V24_PREFLIGHT_OK")
assert "RIZZLER_V24_PREFLIGHT_OK" in (rsp.text or ""), "Gemini preflight failed"
print("Live Gemini preflight PASS; model:",model_short)
(OUT/"gemini_preflight.json").write_text(json.dumps({"passed":True,"model":model_short,"source_sha":SHA},indent=2))

## Strict render dependencies
PDF, PowerPoint and Word strict visual QA require LibreOffice and PyMuPDF. Do not silently turn off strict QA if these are unavailable.

In [ ]:
if shutil.which("libreoffice") is None:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","libreoffice-writer","libreoffice-impress"],check=True)
import fitz
print("Strict visual QA dependencies available:",shutil.which("libreoffice"),"PyMuPDF",fitz.__doc__.splitlines()[0])

## Live end-to-end Work task
This scenario needs grounded research and must create real PPTX, DOCX, PDF and ZIP outputs. No fake model or mock researcher is used. If Gemini or source extraction fails, the result is marked as a failure, not skipped.

In [ ]:
import asyncio
from rizzler.work_store import WorkStore
from rizzler.work_worker import WorkWorker
store=WorkStore(OUT/"work")
owner="kaggle_qualification_owner"
objective=("Investigate the officially documented capabilities and limitations of Google Gemini API grounding, "
           "URL Context and Code Execution. Use verifiable first-party documentation, specify which combinations "
           "require qualification, and create an evidence-linked technical stakeholder PowerPoint, detailed Word "
           "report, PDF report and source package. Do not invent usage limits, model compatibility, or citations.")
job=store.submit(owner,objective,["research","pptx","docx","pdf","zip"],"v24-kaggle-real-gemini-001",[],requested_formats=["pptx","docx","pdf","zip"])
try:
    completed=asyncio.run(WorkWorker(store).once())
except Exception as exc:
    print("Worker raised:",type(exc).__name__)
result=store.get(owner,job["id"])
steps=store.agent_steps(owner,job["id"])
artifacts=store.artifacts(owner,job["id"])
print("Job status:",result["status"],"artifact kinds:",[x["kind"] for x in artifacts])
(OUT/"live_job.json").write_text(json.dumps({"job":result,"steps":steps,"artifacts":artifacts},indent=2,default=str))
assert result["status"]=="completed", "Live job not qualified: inspect evidence bundle"

## Visual, numerical and source QA
This independent pass opens each generated format and checks that every requested artifact exists, is structurally valid and has a digest. It does not substitute for a human factual/visual review.

In [ ]:
from rizzler.artifacts import verify_artifact
artifact_list=store.artifacts(owner,job["id"])
required={"pptx","docx","pdf","zip"}
assert required <= {x["kind"] for x in artifact_list}
report=[]
for record in artifact_list:
    kind=record["kind"]
    path,metadata=store.artifact(owner,record["id"])
    verified=verify_artifact(path,kind)
    assert verified.verified and verified.sha256==record["sha256"]
    print(kind,path)
    report.append({"kind":kind,"artifact_id":record["id"],"sha256":verified.sha256,"verified":True})
(OUT/"artifact_gate.json").write_text(json.dumps({"required":sorted(required),"observed":report},indent=2))
print("Final Work quality:",json.dumps(store.quality_report(owner,job["id"]),indent=2,default=str)[:5000])

## Export full test evidence
The ZIP contains logs, job receipts, qualification decisions and SHA. It does **not** include your Gemini API key.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
target=ROOT/("Rizzler_V24_Qualification_"+SHA[:12]+".zip")
with ZipFile(target,"w",ZIP_DEFLATED) as z:
    for file in OUT.rglob("*"):
        if file.is_file() and "uploads" not in file.parts and file.suffix not in (".sqlite3",".db"):
            z.write(file,file.relative_to(OUT))
print("Evidence ZIP:",target)
print("Source revision:",SHA)